<!-- R01 -->
# Risultati di riferimento · Foundation Vision Studio
Computer Vision, Lezione 18. Questo percorso legge la corsa CPU salvata. Non esegue modelli, non richiede i checkpoint e non completa i tre TODO. In Colab caricare il pacchetto nello stesso runtime quando richiesto. R01–R06 contano tutte le celle.

Nei materiali del corso si usa il **punto decimale** (0.5) e la **virgola seguita da uno spazio** fra componenti di vettori e argomenti, per esempio (1.2, 1, −0.3). I conteggi interi non usano separatori delle migliaia.


**Lessico iniziale.** Feature o embedding: vettore di rappresentazione; patch: regione locale; backbone: estrattore. Prompt: indicazione del target, qui punto o box. Student: rete aggiornata; teacher: rete che fornisce i target; EMA: media mobile dei parametri. Norma L2: lunghezza, usata per normalizzare prima del coseno e della Gram. Support: esempi annotati per costruire i prototipi; query: casi separati da valutare. Few-shot usa un piccolo support; zero-shot richiede di dichiarare l’assenza di esempi per adattamento e l’eventuale assistenza dei prompt oracle.

In [ ]:
# R02
from pathlib import Path
import importlib.util,json,zipfile,hashlib,io
if importlib.util.find_spec('google') and importlib.util.find_spec('google.colab'):
    from google.colab import files
    colab_root=Path('/content/ComputerVision_Lezione18')
    def lesson_recovery():
        return [p for p in colab_root.glob('**/recupero/metriche_finali.json')
                if (p.parent.parent/'manifest.json').is_file()
                and json.loads((p.parent.parent/'manifest.json').read_text()).get('lesson')==18]
    candidates=lesson_recovery()
    if not candidates:
        print('Caricare Pacchetto_studenti_Lezione_18.zip')
        uploaded=files.upload()
        for name,payload in uploaded.items():
            if name.lower().endswith('.zip'):
                with zipfile.ZipFile(io.BytesIO(payload)) as z:
                    assert all(not n.startswith('/') and '..' not in Path(n).parts for n in z.namelist())
                    z.extractall(colab_root)
        candidates=lesson_recovery()
    if len(candidates)!=1:raise RuntimeError('Serve un solo pacchetto valido della Lezione 18 nella cartella Colab dedicata.')
    REC=candidates[0].parent
else:
    base=Path.cwd()
    while not (base/'recupero/metriche_finali.json').exists() and base!=base.parent:base=base.parent
    assert (base/'recupero/metriche_finali.json').exists(),'Aprire dalla cartella notebook/recupero'
    REC=base/'recupero'
manifest=json.loads((REC.parent/'manifest.json').read_text())
assert manifest.get('lesson')==18,'Manifest di una lezione diversa'
for name,digest in manifest['reference_files'].items():
    assert hashlib.sha256((REC/name).read_bytes()).hexdigest()==digest,name
record=json.loads((REC/'metriche_finali.json').read_text())
print('Hash controllati. Risultati importati:',record['device'],record['torch'],record['dinov3_status'])

In [ ]:
# R03
print('IoU importate dalla corsa CPU, non ricalcolate da un modello:')
for row in record['records']:
    print(row['name'],{k:round(v['iou'],6)for k,v in row['semantic'].items()})
print('Tempi originali CPU:',record['cost'])

In [ ]:
# R04
from IPython.display import display,Image
for name in ['recovery_penn14.png','failure_penn22.png','semantic_3.png']:
    display(Image(filename=str(REC/name)))

In [ ]:
# R05
import numpy as np
for row in record['records']:
    with np.load(REC/f"{row['name']}.npz",allow_pickle=False)as masks:
        assert masks['sam_union'].shape==masks['labels'].shape
        print(row['name'],'forme:',masks['sam_union'].shape,'variabili:',masks.files)
print('Le maschere sono leggibili. Per ricalcolare IoU usare la propria C23 completata.')

<!-- R06 -->
## Ritorno alla presentazione
Per nuova inferenza riaprire il notebook studenti e seguire le ripartenze del percorso: C03/C05, funzioni completate e blocchi SAM/LRASPP. Una nuova VM deve preparare pesi e dati. Salvare gli output prima di perderla.